# Chitra Dataset Generator

This notebook demonstrates how to generate the three CSV files required for Chitra chromosome visualization:
1. `ref_chromosome_sizes.csv` - Reference genome chromosome data
2. `species_data.csv` - Species-specific chromosome information
3. `synteny_data.csv` - Synteny/alignment data between species

Plus optional files:
4. `ref_gene_annotations.csv` - Gene annotation data
5. `bp.csv` - Breakpoint data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Run the dataset generator script
%run generate_dataset.py

## Method 1: Generate with default settings

In [ ]:
# Generate dataset with default settings (3 species, all files)
dataset = generate_chitra_dataset()

# Access the generated DataFrames
ref_df = dataset['reference']
species_df = dataset['species']
synteny_df = dataset['synteny']
genes_df = dataset['genes']
breakpoints_df = dataset['breakpoints']

## Method 2: Generate with custom settings

In [ ]:
# Generate dataset with custom settings
custom_dataset = generate_chitra_dataset(
    output_dir="my_custom_dataset",
    num_species=5,  # Generate 5 species instead of 3
    include_optional=True  # Include gene annotations and breakpoints
)

## Explore the Generated Data

In [ ]:
# Display basic information about the generated datasets
print("📊 Reference Chromosomes:")
print(ref_df.head())
print(f"\nShape: {ref_df.shape}")
print(f"Columns: {list(ref_df.columns)}")

In [ ]:
print("📊 Species Data:")
print(species_df.head())
print(f"\nShape: {species_df.shape}")
print(f"Unique species: {species_df['species_name'].unique()}")
print(f"Chromosomes per species:")
print(species_df['species_name'].value_counts())

In [ ]:
print("📊 Synteny Data:")
print(synteny_df.head())
print(f"\nShape: {synteny_df.shape}")
print(f"Strand distribution:")
print(synteny_df['query_strand'].value_counts())
print(f"\nSynteny blocks per species:")
print(synteny_df['query_name'].value_counts())

## Visualize the Data

In [ ]:
# Plot chromosome sizes
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
ref_df.plot(x='chromosome', y='size', kind='bar', ax=plt.gca())
plt.title('Reference Chromosome Sizes')
plt.xticks(rotation=45)
plt.ylabel('Size (bp)')

# Plot synteny block sizes
plt.subplot(1, 2, 2)
synteny_df['block_size'] = synteny_df['ref_end'] - synteny_df['ref_start']
plt.hist(synteny_df['block_size'] / 1e6, bins=20, alpha=0.7)
plt.title('Synteny Block Size Distribution')
plt.xlabel('Block Size (Mb)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Plot species chromosome count and sizes
plt.figure(figsize=(12, 8))

# Chromosome count per species
plt.subplot(2, 2, 1)
species_counts = species_df['species_name'].value_counts()
species_counts.plot(kind='bar')
plt.title('Chromosomes per Species')
plt.ylabel('Number of Chromosomes')
plt.xticks(rotation=45)

# Average chromosome size per species
plt.subplot(2, 2, 2)
avg_sizes = species_df.groupby('species_name')['chr_size_bp'].mean() / 1e6
avg_sizes.plot(kind='bar', color='orange')
plt.title('Average Chromosome Size per Species')
plt.ylabel('Average Size (Mb)')
plt.xticks(rotation=45)

# Synteny strand distribution
plt.subplot(2, 2, 3)
strand_counts = synteny_df['query_strand'].value_counts()
plt.pie(strand_counts.values, labels=strand_counts.index, autopct='%1.1f%%')
plt.title('Synteny Strand Distribution')

# Synteny blocks per species
plt.subplot(2, 2, 4)
synteny_counts = synteny_df['query_name'].value_counts()
synteny_counts.plot(kind='bar', color='green')
plt.title('Synteny Blocks per Species')
plt.ylabel('Number of Blocks')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Validate Data Format

Check that the generated data matches the expected format for Chitra

In [ ]:
# Validate reference chromosome data
print("✅ Validating ref_chromosome_sizes.csv format:")
expected_ref_cols = ['chromosome', 'size', 'centromere_start', 'centromere_end']
print(f"Expected columns: {expected_ref_cols}")
print(f"Actual columns: {list(ref_df.columns)}")
print(f"Columns match: {list(ref_df.columns) == expected_ref_cols}")
print(f"Data types: {ref_df.dtypes.to_dict()}")
print()

In [ ]:
# Validate species data
print("✅ Validating species_data.csv format:")
expected_species_cols = ['species_name', 'chr_id', 'chr_type', 'chr_size_bp', 'centromere_start', 'centromere_end']
print(f"Expected columns: {expected_species_cols}")
print(f"Actual columns: {list(species_df.columns)}")
print(f"Columns match: {list(species_df.columns) == expected_species_cols}")
print(f"Data types: {species_df.dtypes.to_dict()}")
print()

In [ ]:
# Validate synteny data
print("✅ Validating synteny_data.csv format:")
expected_synteny_cols = ['query_name', 'query_chr', 'query_start', 'query_end', 'query_strand', 'ref_chr', 'ref_start', 'ref_end', 'ref_species', 'qry_lvl']
print(f"Expected columns: {expected_synteny_cols}")
print(f"Actual columns: {list(synteny_df.columns)}")
print(f"Columns match: {list(synteny_df.columns) == expected_synteny_cols}")
print(f"Data types: {synteny_df.dtypes.to_dict()}")
print(f"Unique strands: {synteny_df['query_strand'].unique()}")
print()

## Sample Data Preview

Show sample rows from each file to verify the data looks correct

In [ ]:
print("📋 Sample Reference Chromosome Data:")
print(ref_df.head(3).to_string(index=False))
print("\n" + "="*80 + "\n")

print("📋 Sample Species Data:")
print(species_df.head(5).to_string(index=False))
print("\n" + "="*80 + "\n")

print("📋 Sample Synteny Data:")
print(synteny_df.head(5).to_string(index=False))

## Export for Chitra

The files are already saved, but you can also access them programmatically

In [ ]:
# Show file locations
import os
output_dir = "generated_dataset"

print("📁 Generated files ready for Chitra:")
for filename in ['ref_chromosome_sizes.csv', 'species_data.csv', 'synteny_data.csv', 'ref_gene_annotations.csv', 'bp.csv']:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        print(f"  ✅ {filepath} ({size:,} bytes)")
    else:
        print(f"  ❌ {filepath} (not found)")

print(f"\n🎉 Ready to upload to Chitra! Use the files in '{output_dir}' directory.")

## Analyze Synteny Size Variations

The updated script now generates more realistic synteny data where reference and query blocks can have different sizes, simulating real biological scenarios like insertions, deletions, and duplications.

In [ ]:
# Visualize the size variations in synteny blocks
visualize_synteny_variations(synteny_df)

## Examples of Size Variations

Let's look at some specific examples of different size variations:

In [ ]:
# Calculate size ratios
synteny_df['ref_size_mb'] = (synteny_df['ref_end'] - synteny_df['ref_start']) / 1e6
synteny_df['query_size_mb'] = (synteny_df['query_end'] - synteny_df['query_start']) / 1e6
synteny_df['size_ratio'] = synteny_df['query_size_mb'] / synteny_df['ref_size_mb']

print("🔍 Examples of Different Size Variations:\n")

# Similar sizes
similar = synteny_df[(synteny_df['size_ratio'] >= 0.9) & (synteny_df['size_ratio'] <= 1.1)].head(3)
print("📏 Similar sizes (0.9-1.1x ratio):")
for _, row in similar.iterrows():
    print(f"   {row['query_name']} {row['query_chr']}: {row['ref_size_mb']:.2f}Mb → {row['query_size_mb']:.2f}Mb (ratio: {row['size_ratio']:.2f})")

# Query smaller (compression)
compressed = synteny_df[synteny_df['size_ratio'] < 0.7].head(3)
if len(compressed) > 0:
    print("\n🔽 Query compressed (<0.7x ratio):")
    for _, row in compressed.iterrows():
        print(f"   {row['query_name']} {row['query_chr']}: {row['ref_size_mb']:.2f}Mb → {row['query_size_mb']:.2f}Mb (ratio: {row['size_ratio']:.2f})")

# Query larger (expansion)
expanded = synteny_df[synteny_df['size_ratio'] > 1.5].head(3)
if len(expanded) > 0:
    print("\n🔼 Query expanded (>1.5x ratio):")
    for _, row in expanded.iterrows():
        print(f"   {row['query_name']} {row['query_chr']}: {row['ref_size_mb']:.2f}Mb → {row['query_size_mb']:.2f}Mb (ratio: {row['size_ratio']:.2f})")

# Extreme variations
extreme = synteny_df[(synteny_df['size_ratio'] < 0.4) | (synteny_df['size_ratio'] > 2.5)].head(3)
if len(extreme) > 0:
    print("\n⚡ Extreme variations (<0.4x or >2.5x ratio):")
    for _, row in extreme.iterrows():
        variation_type = "massive compression" if row['size_ratio'] < 0.4 else "massive expansion"
        print(f"   {row['query_name']} {row['query_chr']}: {row['ref_size_mb']:.2f}Mb → {row['query_size_mb']:.2f}Mb (ratio: {row['size_ratio']:.2f}) - {variation_type}")